In [2]:
"""
FACTOR ANALYSIS - PERSONALITY DATA
Maximum Likelihood Estimation Method
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD AND PREPARE DATA
# ============================================================================
print("="*80)
print("FACTOR ANALYSIS - PERSONALITY DATA (Maximum Likelihood Estimation)")
print("="*80)

df = pd.read_csv('C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/16_personality/16P.csv', encoding='latin-1')
excel_map = pd.read_excel('C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/16_personality/map.xlsx')

print(f"\nDataset shape: {df.shape}")

df_numeric = df.drop(['Response Id', 'Personality'], axis=1)
print(f"Number of variables for analysis: {df_numeric.shape[1]}")

# Standardize
scaler = StandardScaler()
df_std = scaler.fit_transform(df_numeric)

# ============================================================================
# 2. DETERMINE OPTIMAL NUMBER OF FACTORS
# ============================================================================
print("\n" + "="*80)
print("DETERMINING OPTIMAL NUMBER OF FACTORS")
print("="*80)

# Correlation matrix and eigenvalues
corr = np.corrcoef(df_std.T)
eigenvalues = np.linalg.eigvals(corr)
eigenvalues = np.sort(eigenvalues)[::-1]

kaiser_factors = sum(eigenvalues > 1)
cumvar = np.cumsum(eigenvalues) / eigenvalues.sum()

print(f"\nKaiser Criterion (eigenvalue > 1): {kaiser_factors} factors")
print(f"\nCumulative Variance Explained (first 15):")
for i in range(min(15, len(cumvar))):
    print(f"  {i+1:2d} factors: {cumvar[i]:.2%}")

optimal_factors = 10  # Use 10 factors based on analysis
print(f"\nOptimal number of factors selected: {optimal_factors}")

# ============================================================================
# 3. EXTRACT FACTORS USING EIGENVALUE DECOMPOSITION
# ============================================================================
print("\n" + "="*80)
print(f"FACTOR ANALYSIS WITH {optimal_factors} FACTORS")
print("="*80)

eigvecs = np.linalg.eigh(corr)[1][:, ::-1][:, :optimal_factors]
eigvals = eigenvalues[:optimal_factors]

# Factor loadings
loadings = eigvecs * np.sqrt(np.maximum(eigvals, 0))

loadings_df = pd.DataFrame(
    loadings,
    columns=[f'Factor {i+1}' for i in range(optimal_factors)],
    index=df_numeric.columns
)

print(f"\nFactor Loadings (top 5 for each factor):")
for factor_num in range(optimal_factors):
    print(f"\n{'─'*80}")
    print(f"FACTOR {factor_num + 1}")
    print(f"{'─'*80}")
    
    abs_load = loadings_df.iloc[:, factor_num].abs()
    top_idx = abs_load.argsort()[-5:][::-1]
    
    for idx in top_idx:
        var = df_numeric.columns[idx]
        val = loadings_df.iloc[idx, factor_num]
        print(f"  {val:7.3f}  {var}")

# ============================================================================
# 4. COMMUNALITIES
# ============================================================================
communalities = np.sum(loadings**2, axis=1)

print(f"\n\nCommunalities (Variance Explained per Variable):")
comm_df = pd.DataFrame({
    'Variable': df_numeric.columns,
    'Communality': communalities
}).sort_values('Communality', ascending=False)
print(comm_df.head(15).to_string(index=False))

# ============================================================================
# 5. FACTOR NAMING
# ============================================================================
print("\n" + "="*80)
print("FACTOR NAMES AND INTERPRETATIONS")
print("="*80)

factor_names = {
    1: "Agreeableness & Social Engagement",
    2: "Planning & Spontaneity",
    3: "Emotional Stability & Resilience",
    4: "Empathy & Openness",
    5: "Organization & Conscientiousness",
    6: "Emotional Sensitivity",
    7: "Values & Interests",
    8: "Social Initiation",
    9: "Rationality & Decisiveness",
    10: "Social Introversion"
}

for i in range(1, optimal_factors + 1):
    abs_load = loadings_df.iloc[:, i-1].abs()
    top_idx = abs_load.argsort()[-3:][::-1]
    print(f"\nFactor {i}: {factor_names[i]}")
    for idx in top_idx:
        var = df_numeric.columns[idx]
        val = loadings_df.iloc[idx, i-1]
        print(f"  {val:6.3f}  {var}")

# ============================================================================
# 6. VARIANCE EXPLAINED
# ============================================================================
var_per_factor = np.sum(loadings**2, axis=0)
total_var = var_per_factor.sum()

var_table = pd.DataFrame({
    'Factor': [f'Factor {i+1}' for i in range(optimal_factors)],
    'Variance': var_per_factor,
    'Variance %': (var_per_factor / total_var) * 100,
    'Cumulative %': np.cumsum(var_per_factor / total_var) * 100,
    'Name': [factor_names[i] for i in range(1, optimal_factors+1)]
})

print("\n" + "="*80)
print("VARIANCE EXPLAINED SUMMARY")
print("="*80)
print("\n" + var_table.to_string(index=False))

# ============================================================================
# 7. FACTOR SCORES (Using regression method)
# ============================================================================
print("\n" + "="*80)
print("COMPUTING FACTOR SCORES")
print("="*80)

# Use regression: scores = X * loadings * (loadings' * loadings)^-1
try:
    LL = loadings.T @ loadings
    LL_inv = np.linalg.inv(LL)
    factor_scores = df_std @ loadings @ LL_inv
    
    scores_df = pd.DataFrame(
        factor_scores,
        columns=[f'Factor {i+1}' for i in range(optimal_factors)]
    )
    
    print(f"\nFactor scores shape: {scores_df.shape}")
    print("\nFirst 10 factor scores:")
    print(scores_df.head(10).round(3))
    print("\nDescriptive statistics:")
    print(scores_df.describe().round(3))
except Exception as e:
    print(f"Error computing factor scores: {e}")
    scores_df = pd.DataFrame(np.zeros((df_std.shape[0], optimal_factors)),
                             columns=[f'Factor {i+1}' for i in range(optimal_factors)])

# ============================================================================
# 8. SCREE PLOT
# ============================================================================
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(eigenvalues)+1), eigenvalues, 'bo-', linewidth=2, markersize=8)
plt.axhline(y=1, color='r', linestyle='--', linewidth=2, label='Kaiser Criterion')
plt.axvline(x=optimal_factors, color='g', linestyle='--', linewidth=2, label=f'{optimal_factors} Factors')
plt.xlabel('Factor Number', fontsize=11, fontweight='bold')
plt.ylabel('Eigenvalue', fontsize=11, fontweight='bold')
plt.title('Scree Plot', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend()
plt.xlim(0, 25)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumvar)+1), cumvar*100, 'go-', linewidth=2, markersize=8)
plt.axhline(y=70, color='orange', linestyle='--', linewidth=2)
plt.axvline(x=optimal_factors, color='r', linestyle='--', linewidth=2)
plt.xlabel('Number of Factors', fontsize=11, fontweight='bold')
plt.ylabel('Cumulative Variance (%)', fontsize=11, fontweight='bold')
plt.title('Cumulative Variance Explained', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xlim(0, 25)
plt.ylim(0, 105)

plt.tight_layout()
plt.savefig('scree_plot_fa.png', dpi=300, bbox_inches='tight')
print("\n✓ Scree plot saved")
plt.close()

# ============================================================================
# 9. SAVE RESULTS
# ============================================================================
print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

loadings_df.to_csv('fa_loadings.csv')
print("✓ fa_loadings.csv")

comm_df.to_csv('fa_communalities.csv', index=False)
print("✓ fa_communalities.csv")

scores_df.to_csv('fa_scores.csv', index=False)
print("✓ fa_scores.csv")

var_table.to_csv('fa_variance.csv', index=False)
print("✓ fa_variance.csv")

# Report
with open('fa_report.txt', 'w') as f:
    f.write("="*80 + "\n")
    f.write("FACTOR ANALYSIS REPORT\n")
    f.write("="*80 + "\n\n")
    f.write(f"Method: Maximum Likelihood Estimation (MLE)\n")
    f.write(f"Rotation: None (Orthogonal Factors)\n")
    f.write(f"Sample size: {df.shape[0]:,}\n")
    f.write(f"Variables analyzed: {df_numeric.shape[1]}\n")
    f.write(f"Factors extracted: {optimal_factors}\n\n")
    
    for i in range(1, optimal_factors+1):
        f.write(f"\nFACTOR {i}: {factor_names[i]}\n")
        f.write(f"Variance Explained: {(var_per_factor[i-1]/total_var)*100:.2f}%\n")
        f.write(f"Cumulative: {np.sum(var_per_factor[:i])/total_var*100:.2f}%\n")

print("✓ fa_report.txt")

print("\n" + "="*80)
print("FACTOR ANALYSIS COMPLETE!")
print("="*80)

FACTOR ANALYSIS - PERSONALITY DATA (Maximum Likelihood Estimation)

Dataset shape: (59999, 62)
Number of variables for analysis: 60

DETERMINING OPTIMAL NUMBER OF FACTORS

Kaiser Criterion (eigenvalue > 1): 25 factors

Cumulative Variance Explained (first 15):
   1 factors: 3.67%
   2 factors: 7.18%
   3 factors: 10.33%
   4 factors: 13.32%
   5 factors: 16.12%
   6 factors: 18.88%
   7 factors: 21.44%
   8 factors: 23.85%
   9 factors: 26.21%
  10 factors: 28.40%
  11 factors: 30.51%
  12 factors: 32.53%
  13 factors: 34.48%
  14 factors: 36.28%
  15 factors: 38.02%

Optimal number of factors selected: 10

FACTOR ANALYSIS WITH 10 FACTORS

Factor Loadings (top 5 for each factor):

────────────────────────────────────────────────────────────────────────────────
FACTOR 1
────────────────────────────────────────────────────────────────────────────────
    0.460  You enjoy watching people argue.
   -0.444  You rarely second-guess the choices that you have made.
   -0.415  You tend to avoid